# Task 3 — Dual Local Models Blind A/B (Google Colab / Jupyter)

Этот блокнот — **специальная версия Task 3** для честного сравнения двух локальных моделей на одной и той же входной выборке.

Он делает следующее:

- принимает вход из **query**, **списка DOI / URL / identifier**, **commands** или **Task 1 YAML**;
- запускает **один и тот же Task 3 pipeline два раза**:
  - для **локальной модели из коробки**,
  - для **локальной дообученной модели**;
- старается использовать **одни и те же processed papers**, чтобы сравнение было максимально честным;
- собирает **blind offline A/B form**, где эксперт видит только **Variant A / Variant B** и **Hidden model α / Hidden model β**;
- сохраняет:
  - `task3_dual_local_model_review_offline_ab.html`
  - `expert_dual_model_blind_review_bundle.zip`
  - `task3_dual_local_model_blind_key.json` (**owner-only**, не передавать эксперту)

> Это обычный Jupyter notebook, который можно открывать как в Colab, так и в совместимых Jupyter-средах.


# Пошаговый туториал

## Базовый сценарий
1. Запустите ячейку **«Установка и импорт»**.
2. Заполните входные данные:
   - `Query + identifiers`,
   - `Task 1 YAML`,
   - или `Commands`.
3. Укажите **две локальные модельные конфигурации**:
   - **Model α** — обычно baseline / model out-of-the-box;
   - **Model β** — обычно finetuned local model.
4. При необходимости включите **HF offline mode**, если модели лежат локально или уже закешированы.
5. Запустите нижнюю ячейку **«Запуск dual-local blind A/B»**.
6. На выходе получите:
   - два отдельных Task 3 bundle,
   - blind offline HTML для эксперта,
   - owner-only key file,
   - expert ZIP без раскрытия identity key.

## Почему этот workflow честнее обычного
- Сначала выполняется **run α**.
- Затем **run β** использует те же `processed_papers`, чтобы не скачивать статьи заново и не менять входные данные.
- Blind review не показывает эксперту, какая именно модель стоит за α / β.

## Формат commands
Можно вставить текст или загрузить `.txt / .yaml / .json`, например:

```text
query: temporal knowledge graph multimodal hypothesis generation
identifier: 10.1038/s41586-020-2649-2
identifier: https://doi.org/10.1126/science.abc1234
domain_id: science
top_papers: 12
top_hypotheses: 8
candidate_top_k: 16
top_pairs: 8
edge_mode: auto
link_prediction_backend: auto
hf_offline: true
model_a_owner_label: base_local_model
model_a_vlm_backend: qwen2_vl
model_a_vlm_model_id: /models/qwen2.5-vl-base
model_a_local_text_model: llama3.1:8b
model_b_owner_label: finetuned_local_model
model_b_vlm_backend: qwen2_vl
model_b_vlm_model_id: /models/qwen2.5-vl-finetuned
model_b_local_text_model: llama3.1:8b
```

## Что передавать эксперту
Эксперту нужно передавать **offline HTML** и/или **expert ZIP**.

**Не передавайте эксперту файл `task3_dual_local_model_blind_key.json`** — это owner-side mapping между анонимными системами и реальными моделями.


In [ ]:
# @title
# Установка и импорт (запустите эту ячейку)
import gc, io, json, os, sys, tempfile, subprocess, zipfile, shutil
from pathlib import Path

os.environ.setdefault('G4F_ASYNC_ENABLED', '1')
os.environ.setdefault('G4F_ASYNC_MAX_CONCURRENCY', '3')
os.environ.setdefault('G4F_ASYNC_RETRIES', '3')
os.environ.setdefault('G4F_ASYNC_MAX_MODELS_PER_REQUEST', '3')
os.environ.setdefault('LLM_REQUEST_TIMEOUT_SECONDS', '25')
os.environ.setdefault('VLM_REQUEST_TIMEOUT_SECONDS', '45')

if os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE') == '1':
    os.environ.setdefault('LLM_PROVIDER', 'mock')
    os.environ.setdefault('LLM_MODEL', 'mock')
    os.environ.setdefault('EMBED_PROVIDER', 'hash')
    os.environ.setdefault('MM_EMBED_BACKEND', 'none')
    os.environ.setdefault('VLM_BACKEND', 'none')
    os.environ.setdefault('HF_HUB_OFFLINE', '1')
    os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
    os.environ.setdefault('HF_DATASETS_OFFLINE', '1')



def run(cmd, cwd=None):
    print('>', ' '.join(map(str, cmd)))
    subprocess.check_call(cmd, cwd=cwd)


def run_optional(cmd, cwd=None, label=None):
    try:
        run(cmd, cwd=cwd)
        return True
    except Exception as e:
        label = label or ' '.join(map(str, cmd))
        print(f'[warn] optional step failed: {label}: {type(e).__name__}: {e}')
        return False


def _materialize_local_repo_archive() -> Path | None:
    search_roots = [Path('/mnt/data'), Path('/content'), Path.cwd(), Path.cwd().parent]
    patterns = (
        'top-papers-graph-main-task3-module.zip',
        'top-papers-graph-main-task3-notebook.zip',
        'top-papers-graph-main.zip',
        'top-papers-graph*.zip',
        '*top-papers-graph*.zip',
    )
    for base in search_roots:
        if not base.exists():
            continue
        for pattern in patterns:
            for archive in sorted(base.glob(pattern)):
                target = archive.with_suffix('')
                try:
                    candidate_root = target / 'top-papers-graph-main'
                    if (candidate_root / 'pyproject.toml').exists() and (candidate_root / 'src' / 'scireason').exists():
                        return candidate_root
                    if target.exists() and (target / 'pyproject.toml').exists() and (target / 'src' / 'scireason').exists():
                        return target
                    target.mkdir(parents=True, exist_ok=True)
                    with zipfile.ZipFile(archive, 'r') as zf:
                        zf.extractall(target)
                    if (candidate_root / 'pyproject.toml').exists() and (candidate_root / 'src' / 'scireason').exists():
                        return candidate_root
                    if (target / 'pyproject.toml').exists() and (target / 'src' / 'scireason').exists():
                        return target
                except Exception as e:
                    print(f'[warn] local repo archive skipped: {archive}: {type(e).__name__}: {e}')
    return None


def find_repo_root() -> Path | None:
    env_dir = os.environ.get('TPG_REPO_DIR')
    if env_dir:
        env_path = Path(env_dir)
        if (env_path / 'pyproject.toml').exists() and (env_path / 'src' / 'scireason').exists():
            return env_path

    candidates = []
    repo_from_archive = _materialize_local_repo_archive()
    if repo_from_archive is not None:
        candidates.append(repo_from_archive)
    if env_dir:
        candidates.append(Path(env_dir))
    cwd = Path.cwd()
    candidates.extend([cwd, cwd.parent, Path('/content'), Path('/mnt/data')])
    search_bases = [Path('/content'), Path('/mnt/data'), cwd, cwd.parent]
    patterns = ('top-papers-graph-main*', 'top-papers-graph*')
    for base in search_bases:
        if not base.exists():
            continue
        for pattern in patterns:
            candidates.extend(base.glob(pattern))
    seen = set()
    uniq = []
    for c in candidates:
        try:
            r = c.resolve()
        except Exception:
            r = c
        key = str(r)
        if key in seen:
            continue
        seen.add(key)
        uniq.append(r)
    for c in uniq:
        if (c / 'pyproject.toml').exists() and (c / 'src' / 'scireason').exists():
            return c
    return None


REPO_DIR = find_repo_root()
REPO_URL = 'https://github.com/top-papers/top-papers-graph.git'
if REPO_DIR is None:
    REPO_DIR = Path('/content/top-papers-graph') if Path('/content').exists() else Path.cwd() / 'top-papers-graph'
    if not REPO_DIR.exists():
        run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])

SRC_DIR = REPO_DIR / 'src'
if not SRC_DIR.exists():
    raise FileNotFoundError(f'Не найден каталог src в репозитории: {REPO_DIR}')
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

run_optional([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets', 'pyyaml', 'requests', 'unidecode', 'nbformat'])
_task3_editable_install = [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[task3]']
if os.environ.get('TASK3_NOTEBOOK_SMOKE') == '1' or os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE') == '1' or os.environ.get('TASK3_NOTEBOOK_SKIP_OPTIONAL_DEPS') == '1':
    _task3_editable_install = [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps']
run_optional(_task3_editable_install, cwd=REPO_DIR, label='pip install editable task3 notebook runtime')

import yaml
import ipywidgets as W
from IPython.display import HTML, Markdown, display, clear_output
from unidecode import unidecode

from scireason.config import settings
from scireason.pipeline.task3_hypothesis_generation import prepare_task3_hypothesis_bundle
from scireason.task3_dual_model_review import (
    build_task3_dual_model_expert_bundle,
    build_task3_dual_model_offline_review_package,
)

TASK3_DEFAULT_LOCAL_VLM_MODEL = getattr(settings, 'vlm_model_id', 'Qwen/Qwen2.5-VL-3B-Instruct') or 'Qwen/Qwen2.5-VL-3B-Instruct'
TASK3_DUAL_LAST_RESULT = None
TASK3_DUAL_LAST_TASK_META = None
TASK3_DUAL_LAST_ARTIFACTS = None

print('REPO_DIR =', REPO_DIR)
print('SRC_DIR  =', SRC_DIR)
print('task3 default local VLM model =', TASK3_DEFAULT_LOCAL_VLM_MODEL)


In [ ]:
# @title
# Вспомогательные функции (не нужно редактировать)
import hashlib
import json
import re
from pathlib import Path


def _escape_html(value):
    return html_escape(str(value or ''))


def html_escape(text: str) -> str:
    return (
        str(text or '')
        .replace('&', '&amp;')
        .replace('<', '&lt;')
        .replace('>', '&gt;')
        .replace('"', '&quot;')
        .replace("'", '&#39;')
    )


def _slugify(text: str) -> str:
    value = unidecode(str(text or '')).lower()
    value = re.sub(r'[^a-z0-9\s_-]+', '', value)
    value = re.sub(r'\s+', '_', value).strip('_')
    return value or 'expert'


def _materialize_upload(upload_value, out_dir: Path, default_name: str) -> Path | None:
    if not upload_value:
        return None
    out_dir.mkdir(parents=True, exist_ok=True)
    if isinstance(upload_value, dict):
        if not upload_value:
            return None
        name, meta = next(iter(upload_value.items()))
        content = meta.get('content')
        if content is None:
            return None
        path = out_dir / (name or default_name)
        path.write_bytes(content if isinstance(content, (bytes, bytearray)) else bytes(content))
        return path
    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        content = meta.get('content')
        if content is None:
            return None
        name = meta.get('name') or default_name
        path = out_dir / name
        path.write_bytes(content if isinstance(content, (bytes, bytearray)) else bytes(content))
        return path
    raise TypeError('Неподдерживаемый формат upload value')


def _read_upload_name_and_bytes(upload_value):
    if not upload_value:
        return None, None
    if isinstance(upload_value, dict) and upload_value:
        name, meta = next(iter(upload_value.items()))
        return name or 'upload.bin', meta.get('content')
    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        return meta.get('name') or 'upload.bin', meta.get('content')
    return None, None


def _parse_bool(value, default=None):
    if value is None:
        return default
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {'1', 'true', 'yes', 'y', 'on', 'да'}:
        return True
    if text in {'0', 'false', 'no', 'n', 'off', 'нет'}:
        return False
    return default


def _parse_identifier_blob(text: str):
    parts = []
    for raw_line in str(text or '').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        for token in re.split(r'[,;]', line):
            token = token.strip()
            if token:
                parts.append(token)
    return parts


def _parse_commands_payload(text: str) -> dict:
    payload = {
        'query': '',
        'identifiers': [],
        'domain_id': None,
        'top_papers': None,
        'top_hypotheses': None,
        'candidate_top_k': None,
        'search_limit': None,
        'top_pairs': None,
        'edge_mode': None,
        'link_prediction_backend': None,
        'include_multimodal': None,
        'run_vlm': None,
        'hf_offline': None,
        'processed_dir': None,
        'out_dir': None,
        'model_a_owner_label': None,
        'model_a_vlm_backend': None,
        'model_a_vlm_model_id': None,
        'model_a_local_text_model': None,
        'model_b_owner_label': None,
        'model_b_vlm_backend': None,
        'model_b_vlm_model_id': None,
        'model_b_local_text_model': None,
    }
    text = str(text or '').strip()
    if not text:
        return payload

    for parser_name, parser in [('json', json.loads), ('yaml', yaml.safe_load)]:
        try:
            obj = parser(text)
            if isinstance(obj, dict):
                for key in list(payload.keys()):
                    if key == 'identifiers':
                        continue
                    if key in obj:
                        payload[key] = obj[key]
                if 'identifier' in obj:
                    payload['identifiers'].extend(_parse_identifier_blob(obj['identifier']))
                if 'identifiers' in obj:
                    if isinstance(obj['identifiers'], list):
                        payload['identifiers'].extend([str(x).strip() for x in obj['identifiers'] if str(x).strip()])
                    else:
                        payload['identifiers'].extend(_parse_identifier_blob(obj['identifiers']))
                return payload
        except Exception:
            pass

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if ':' not in line:
            payload['identifiers'].append(line)
            continue
        key, value = line.split(':', 1)
        key = key.strip().lower()
        value = value.strip()
        if key in {'query', 'topic'}:
            payload['query'] = value
        elif key in {'identifier', 'id', 'doi', 'url'}:
            payload['identifiers'].extend(_parse_identifier_blob(value))
        elif key in {'identifiers', 'ids', 'doi_list', 'url_list'}:
            payload['identifiers'].extend(_parse_identifier_blob(value))
        elif key in {'domain', 'domain_id'}:
            payload['domain_id'] = value
        elif key in {'top_papers', 'top_hypotheses', 'candidate_top_k', 'search_limit', 'top_pairs'}:
            try:
                payload[key] = int(value)
            except Exception:
                pass
        elif key in {'edge_mode', 'link_prediction_backend', 'processed_dir', 'out_dir',
                     'model_a_owner_label', 'model_a_vlm_backend', 'model_a_vlm_model_id', 'model_a_local_text_model',
                     'model_b_owner_label', 'model_b_vlm_backend', 'model_b_vlm_model_id', 'model_b_local_text_model'}:
            payload[key] = value
        elif key in {'include_multimodal', 'run_vlm', 'hf_offline'}:
            payload[key] = _parse_bool(value, default=None)
    return payload


def _merged_commands_from_widgets(commands_text_value: str, commands_upload_value) -> dict:
    merged = _parse_commands_payload(commands_text_value)
    upload_name, upload_bytes = _read_upload_name_and_bytes(commands_upload_value)
    if upload_bytes:
        text = upload_bytes.decode('utf-8', errors='replace') if isinstance(upload_bytes, (bytes, bytearray)) else bytes(upload_bytes).decode('utf-8', errors='replace')
        uploaded = _parse_commands_payload(text)
        for key, value in uploaded.items():
            if key == 'identifiers':
                merged['identifiers'].extend([x for x in value if x])
            elif value not in (None, '', []):
                merged[key] = value
    deduped = []
    seen = set()
    for item in merged['identifiers']:
        norm = str(item).strip()
        if not norm or norm in seen:
            continue
        seen.add(norm)
        deduped.append(norm)
    merged['identifiers'] = deduped
    return merged


def _load_yaml_if_present(path: Path | None) -> dict:
    if path is None or not path.exists():
        return {}
    data = yaml.safe_load(path.read_text(encoding='utf-8')) or {}
    if isinstance(data, dict):
        return data
    raise ValueError('Ожидался YAML с top-level object')


def _discover_processed_dir(root: Path) -> Path | None:
    candidates = []
    for p in [root, *root.rglob('*')]:
        if not p.is_dir():
            continue
        has_meta = any(child.is_file() and child.name == 'meta.json' for child in p.glob('*/meta.json'))
        has_chunks = any(child.is_file() and child.name == 'chunks.jsonl' for child in p.glob('*/chunks.jsonl'))
        if has_meta or has_chunks:
            candidates.append(p)
    if candidates:
        candidates.sort(key=lambda p: len(p.parts))
        return candidates[0]
    return None


def _extract_processed_zip(upload_value, workdir: Path) -> Path | None:
    archive_path = _materialize_upload(upload_value, workdir, 'processed_papers.zip')
    if archive_path is None:
        return None
    extract_dir = workdir / 'processed_from_zip'
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path, 'r') as zf:
        zf.extractall(extract_dir)
    found = _discover_processed_dir(extract_dir)
    if found is None:
        raise FileNotFoundError('Не удалось найти processed_papers внутри ZIP-архива')
    return found


def _task3_task_meta_from_widgets(base_doc: dict | None = None):
    base_doc = dict(base_doc or {})
    expert = base_doc.get('expert') if isinstance(base_doc.get('expert'), dict) else {}
    last_name = str(expert_last.value or expert.get('last_name') or '').strip()
    first_name = str(expert_first.value or expert.get('first_name') or '').strip()
    patronymic = str(expert_pat.value or expert.get('patronymic') or '-').strip() or '-'
    full_name = ' '.join(x for x in [last_name, first_name, patronymic] if x).strip()
    latin_slug = _slugify(full_name)

    query_candidate = str(query.value or '').strip()
    submission_id = str(base_doc.get('submission_id') or '').strip()
    if not submission_id:
        seed_text = query_candidate or str(base_doc.get('topic') or 'task3_dual_local')
        short_hash = hashlib.sha256(seed_text.encode('utf-8')).hexdigest()[:12]
        submission_id = f'{latin_slug or "expert"}__{short_hash}'

    topic_value = str(base_doc.get('topic') or query_candidate or '').strip()
    return {
        'topic': topic_value,
        'submission_id': submission_id,
        'cutoff_year': str(base_doc.get('cutoff_year') or '').strip(),
        'domain': str(base_doc.get('domain') or domain_id.value or '').strip(),
        'expert': {
            'last_name': last_name,
            'first_name': first_name,
            'patronymic': patronymic,
            'full_name': full_name,
            'latin_slug': latin_slug,
        },
    }


def _download_file_if_possible(path: Path, status_widget=None):
    try:
        from google.colab import files as colab_files
        colab_files.download(str(path))
        return True
    except Exception:
        if status_widget is not None:
            current = str(getattr(status_widget, 'value', '') or '')
            status_widget.value = current + f'<div><code>{_escape_html(str(path))}</code></div>'
        return False


def _top_hypotheses_summary(hypotheses_path: Path, prefix: str = 'H', limit: int = 5) -> str:
    try:
        rows = json.loads(hypotheses_path.read_text(encoding='utf-8'))
    except Exception:
        return 'Не удалось прочитать hypotheses_ranked.json'
    if not isinstance(rows, list) or not rows:
        return 'Гипотезы не были сгенерированы.'
    lines = []
    for row in rows[:limit]:
        hyp = row.get('hypothesis') if isinstance(row.get('hypothesis'), dict) else {}
        title = str(hyp.get('title') or '(без названия)')
        lines.append(f"- **{prefix}-{int(row.get('rank') or 0):03d}** — {title}  ")
    return "\n".join(lines)

In [ ]:
# @title
# Форма dual-local blind A/B (запустите ячейку и заполните поля)
from IPython.display import display

input_mode = W.Dropdown(
    options=[
        ('Task 1 YAML', 'yaml'),
        ('Query + identifiers', 'query'),
        ('Commands', 'commands'),
    ],
    value='query',
    description='Input mode',
)

trajectory_upload = W.FileUpload(accept='.yaml,.yml', multiple=False, description='Task1 YAML')
processed_upload = W.FileUpload(accept='.zip', multiple=False, description='Processed ZIP')
commands_upload = W.FileUpload(accept='.txt,.yaml,.yml,.json', multiple=False, description='Commands file')

query = W.Textarea(
    value='',
    description='Query',
    placeholder='temporal knowledge graph multimodal hypothesis generation',
    layout=W.Layout(width='100%', height='90px'),
)
identifiers_text = W.Textarea(
    value='',
    description='IDs',
    placeholder='DOI / URL / identifier — по одному на строку',
    layout=W.Layout(width='100%', height='110px'),
)
commands_text = W.Textarea(
    value='',
    description='Commands',
    placeholder="query: ...\nidentifier: ...\nmodel_a_vlm_model_id: /models/base\nmodel_b_vlm_model_id: /models/tuned\n...",
    layout=W.Layout(width='100%', height='170px'),
)

expert_last = W.Text(description='Фамилия', placeholder='Иванов', layout=W.Layout(width='100%'))
expert_first = W.Text(description='Имя', placeholder='Иван', layout=W.Layout(width='100%'))
expert_pat = W.Text(description='Отчество', placeholder='Иванович или -', value='-', layout=W.Layout(width='100%'))

domain_id = W.Text(value='science', description='Domain', layout=W.Layout(width='100%'))
out_dir = W.Text(value='runs/task3_dual_local_blind_ab', description='Out dir', layout=W.Layout(width='100%'))
search_limit = W.IntSlider(value=25, min=5, max=100, step=5, description='Search limit', continuous_update=False)
top_papers = W.IntSlider(value=12, min=0, max=50, step=1, description='Top papers', continuous_update=False)
top_hypotheses = W.IntSlider(value=8, min=1, max=20, step=1, description='Top hyps', continuous_update=False)
candidate_top_k = W.IntSlider(value=16, min=4, max=50, step=1, description='Cand top-k', continuous_update=False)
top_pairs = W.IntSlider(value=8, min=1, max=20, step=1, description='Top pairs', continuous_update=False)
include_multimodal = W.Checkbox(value=True, description='Include multimodal chunks')
run_vlm = W.Checkbox(value=True, description='Run VLM on pages / images / tables')
edge_mode = W.Dropdown(options=['auto', 'cooccurrence', 'paper_chain'], value='auto', description='Edge mode')
link_prediction_backend = W.Dropdown(options=['auto', 'heuristic', 'pygt_temporal', 'tgn'], value='auto', description='Link pred')
annoy_n_trees = W.IntSlider(value=32, min=4, max=64, step=4, description='Annoy trees', continuous_update=False)
annoy_top_k = W.IntSlider(value=6, min=1, max=20, step=1, description='Annoy top-k', continuous_update=False)

model_a_owner_label = W.Text(value='base_local_model', description='Owner A', layout=W.Layout(width='100%'))
model_a_vlm_backend = W.Dropdown(
    options=[
        ('qwen2_vl (HF local)', 'qwen2_vl'),
        ('qwen3_vl (HF local)', 'qwen3_vl'),
        ('llava (HF local)', 'llava'),
        ('phi3_vision (HF local)', 'phi3_vision'),
        ('auto', 'auto'),
        ('none', 'none'),
    ],
    value='qwen2_vl',
    description='VLM α',
)
model_a_vlm_model_id = W.Text(value=TASK3_DEFAULT_LOCAL_VLM_MODEL, description='Model/path α', placeholder='Qwen/Qwen2.5-VL-3B-Instruct или /models/base-vlm', layout=W.Layout(width='100%'))
model_a_local_text_model = W.Text(value='', description='Text α', placeholder='необязательно: локальная text model', layout=W.Layout(width='100%'))

model_b_owner_label = W.Text(value='finetuned_local_model', description='Owner β', layout=W.Layout(width='100%'))
model_b_vlm_backend = W.Dropdown(
    options=[
        ('qwen2_vl (HF local)', 'qwen2_vl'),
        ('qwen3_vl (HF local)', 'qwen3_vl'),
        ('llava (HF local)', 'llava'),
        ('phi3_vision (HF local)', 'phi3_vision'),
        ('auto', 'auto'),
        ('none', 'none'),
    ],
    value='qwen2_vl',
    description='VLM β',
)
model_b_vlm_model_id = W.Text(value='', description='Model/path β', placeholder='/models/finetuned-vlm', layout=W.Layout(width='100%'))
model_b_local_text_model = W.Text(value='', description='Text β', placeholder='необязательно: локальная text model', layout=W.Layout(width='100%'))

hf_offline = W.Checkbox(value=False, description='HF offline / local-files mode')
create_offline_form = W.Checkbox(value=True, description='Build blind offline HTML')
create_expert_bundle = W.Checkbox(value=True, description='Build expert ZIP (without owner key)')
auto_download_offline = W.Checkbox(value=False, description='Auto-download offline HTML (Colab)')
auto_download_bundle = W.Checkbox(value=False, description='Auto-download expert ZIP (Colab)')
auto_download_owner_key = W.Checkbox(value=False, description='Auto-download owner key (Colab)')

status = W.HTML()
summary_html = W.HTML()
out = W.Output(layout=W.Layout(border='1px solid var(--jp-border-color2, #e2e8f0)', padding='8px'))

last_paths_html = W.HTML('<div class="task3-note"><b>Артефакты ещё не созданы.</b></div>')
download_offline_btn = W.Button(description='Скачать offline HTML', button_style='primary')
download_offline_btn.disabled = True
download_bundle_btn = W.Button(description='Скачать expert ZIP', button_style='warning')
download_bundle_btn.disabled = True
download_owner_key_btn = W.Button(description='Скачать owner key', button_style='info')
download_owner_key_btn.disabled = True
artifact_out = W.Output()


def _set_task3dual_paths(paths):
    global TASK3_DUAL_LAST_ARTIFACTS
    TASK3_DUAL_LAST_ARTIFACTS = paths
    offline_p = paths.get('offline_form') if isinstance(paths, dict) else None
    bundle_p = paths.get('expert_bundle') if isinstance(paths, dict) else None
    owner_p = paths.get('owner_key') if isinstance(paths, dict) else None
    download_offline_btn.disabled = not (offline_p and Path(offline_p).exists())
    download_bundle_btn.disabled = not (bundle_p and Path(bundle_p).exists())
    download_owner_key_btn.disabled = not (owner_p and Path(owner_p).exists())
    parts = []
    if paths:
        for key in [
            'run_alpha_bundle', 'run_beta_bundle', 'manifest_alpha', 'manifest_beta',
            'hypotheses_alpha_json', 'hypotheses_beta_json', 'offline_form', 'expert_bundle', 'owner_key', 'public_manifest'
        ]:
            if paths.get(key):
                parts.append(f'<div><b>{key}</b>: <code>{_escape_html(str(paths[key]))}</code></div>')
    last_paths_html.value = '<div class="task3-note">' + ''.join(parts) + '</div>' if parts else '<div class="task3-note">Нет артефактов.</div>'


def _download_offline(_):
    artifact_out.clear_output()
    paths = TASK3_DUAL_LAST_ARTIFACTS or {}
    target = Path(paths.get('offline_form')) if paths.get('offline_form') else None
    if target is None or not target.exists():
        status.value = '<div><b>Offline HTML пока не готов.</b></div>'
        return
    _download_file_if_possible(target, status_widget=status)


def _download_bundle(_):
    artifact_out.clear_output()
    paths = TASK3_DUAL_LAST_ARTIFACTS or {}
    target = Path(paths.get('expert_bundle')) if paths.get('expert_bundle') else None
    if target is None or not target.exists():
        status.value = '<div><b>Expert ZIP пока не готов.</b></div>'
        return
    _download_file_if_possible(target, status_widget=status)


def _download_owner_key(_):
    artifact_out.clear_output()
    paths = TASK3_DUAL_LAST_ARTIFACTS or {}
    target = Path(paths.get('owner_key')) if paths.get('owner_key') else None
    if target is None or not target.exists():
        status.value = '<div><b>Owner key пока не готов.</b></div>'
        return
    _download_file_if_possible(target, status_widget=status)


download_offline_btn.on_click(_download_offline)
download_bundle_btn.on_click(_download_bundle)
download_owner_key_btn.on_click(_download_owner_key)

left = W.VBox([
    W.HTML('<h3>Входные данные</h3>'),
    input_mode,
    trajectory_upload,
    processed_upload,
    query,
    identifiers_text,
    commands_text,
    commands_upload,
])
right = W.VBox([
    W.HTML('<h3>Эксперт и общие параметры</h3>'),
    expert_last,
    expert_first,
    expert_pat,
    domain_id,
    out_dir,
    search_limit,
    top_papers,
    top_hypotheses,
    candidate_top_k,
    top_pairs,
    include_multimodal,
    run_vlm,
    edge_mode,
    link_prediction_backend,
    annoy_n_trees,
    annoy_top_k,
])
model_alpha_box = W.VBox([
    W.HTML('<h3>Model α (обычно baseline / из коробки)</h3>'),
    model_a_owner_label,
    model_a_vlm_backend,
    model_a_vlm_model_id,
    model_a_local_text_model,
])
model_beta_box = W.VBox([
    W.HTML('<h3>Model β (обычно finetuned)</h3>'),
    model_b_owner_label,
    model_b_vlm_backend,
    model_b_vlm_model_id,
    model_b_local_text_model,
])
common_box = W.VBox([
    W.HTML('<h3>Blind review и сохранение</h3>'),
    hf_offline,
    create_offline_form,
    create_expert_bundle,
    auto_download_offline,
    auto_download_bundle,
    auto_download_owner_key,
])

ui = W.VBox([
    W.HBox([left, right], layout=W.Layout(align_items='flex-start')),
    W.HBox([model_alpha_box, model_beta_box, common_box], layout=W.Layout(align_items='flex-start')),
    status,
    summary_html,
    W.HBox([download_offline_btn, download_bundle_btn, download_owner_key_btn]),
    last_paths_html,
    artifact_out,
    out,
])

# Headless smoke defaults for automated validation
if os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE') == '1':
    _smoke_query = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_QUERY') or 'offline dual local model blind review').strip()
    _smoke_processed = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_PROCESSED_DIR') or '').strip()
    _smoke_out_dir = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_OUT_DIR') or 'runs/task3_dual_local_blind_ab_smoke').strip()
    _smoke_model_a = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_MODEL_A') or 'mock-local-model-a').strip() or 'mock-local-model-a'
    _smoke_model_b = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_MODEL_B') or 'mock-local-model-b').strip() or 'mock-local-model-b'

    input_mode.value = 'commands'
    query.value = _smoke_query
    commands_text.value = (
        f"query: {_smoke_query}\n"
        + (f"processed_dir: {_smoke_processed}\n" if _smoke_processed else "")
        + f"out_dir: {_smoke_out_dir}\n"
        + "top_papers: 0\n"
        + "top_hypotheses: 3\n"
        + "candidate_top_k: 5\n"
        + "top_pairs: 3\n"
        + "include_multimodal: true\n"
        + "run_vlm: false\n"
        + "edge_mode: cooccurrence\n"
        + "link_prediction_backend: heuristic\n"
        + "hf_offline: true\n"
        + "model_a_vlm_backend: none\n"
        + f"model_a_vlm_model_id: {_smoke_model_a}\n"
        + "model_b_vlm_backend: none\n"
        + f"model_b_vlm_model_id: {_smoke_model_b}\n"
    )
    expert_last.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_EXPERT_LAST', 'Smoke')
    expert_first.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_EXPERT_FIRST', 'Validation')
    expert_pat.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_EXPERT_PAT', '-')
    domain_id.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_DOMAIN_ID', 'science')
    out_dir.value = _smoke_out_dir
    search_limit.value = 5
    top_papers.value = 0
    top_hypotheses.value = 3
    candidate_top_k.value = 5
    top_pairs.value = 3
    include_multimodal.value = True
    run_vlm.value = False
    edge_mode.value = 'cooccurrence'
    link_prediction_backend.value = 'heuristic'
    model_a_owner_label.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_OWNER_A', 'base_local_model')
    model_a_vlm_backend.value = 'none'
    model_a_vlm_model_id.value = _smoke_model_a
    model_a_local_text_model.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_TEXT_A', '')
    model_b_owner_label.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_OWNER_B', 'finetuned_local_model')
    model_b_vlm_backend.value = 'none'
    model_b_vlm_model_id.value = _smoke_model_b
    model_b_local_text_model.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_TEXT_B', '')
    hf_offline.value = True
    create_offline_form.value = True
    create_expert_bundle.value = True
    auto_download_offline.value = False
    auto_download_bundle.value = False
    auto_download_owner_key.value = False
    status.value = '<div><b>Smoke mode.</b> Notebook prefilled from TASK3_DUAL_NOTEBOOK_SMOKE_* env vars.</div>'

display(ui)

In [ ]:
# @title
# Запуск dual-local blind A/B из отдельной ячейки
# Важно: после заполнения формы выше запускайте именно ЭТУ ячейку.


def run_task3_dual_local_ab_from_form():
    global TASK3_DUAL_LAST_RESULT, TASK3_DUAL_LAST_TASK_META
    with out:
        clear_output()
        try:
            status.value = '<div><b>Task 3 dual-local blind A/B: подготовка запуска...</b></div>'
            workdir = Path(tempfile.mkdtemp(prefix='task3_dual_local_ab_'))

            trajectory_path = _materialize_upload(trajectory_upload.value, workdir, 'task1.yaml')
            commands_file_path = _materialize_upload(commands_upload.value, workdir, 'commands.txt')
            processed_dir = _extract_processed_zip(processed_upload.value, workdir)

            commands = _merged_commands_from_widgets(commands_text.value, commands_upload.value)
            trajectory_doc = _load_yaml_if_present(trajectory_path) if trajectory_path else {}
            trajectory_for_run = trajectory_path if input_mode.value == 'yaml' else None
            task_meta = _task3_task_meta_from_widgets(trajectory_doc if trajectory_for_run else {})
            TASK3_DUAL_LAST_TASK_META = task_meta

            inline_identifiers = _parse_identifier_blob(identifiers_text.value)
            identifiers = list(inline_identifiers)
            identifiers.extend(commands.get('identifiers') or [])
            deduped = []
            seen = set()
            for item in identifiers:
                item = str(item).strip()
                if not item or item in seen:
                    continue
                seen.add(item)
                deduped.append(item)
            identifiers = deduped

            effective_query = str(query.value or '').strip()
            if not effective_query:
                effective_query = str(commands.get('query') or '').strip()
            if not effective_query and trajectory_doc:
                effective_query = str(trajectory_doc.get('topic') or '').strip()
            if not effective_query and processed_dir is not None:
                effective_query = 'offline dual local model blind review'

            if input_mode.value == 'yaml' and trajectory_path is None:
                raise ValueError('Выбран режим Task 1 YAML, но YAML-файл не загружен.')
            if input_mode.value == 'commands' and not (commands_text.value.strip() or commands_file_path):
                raise ValueError('Выбран режим Commands, но commands text/file пустой.')
            if input_mode.value == 'query' and not effective_query and not identifiers and processed_dir is None:
                raise ValueError('Введите query и/или identifiers, либо загрузите processed ZIP.')

            if processed_dir is None and commands.get('processed_dir'):
                candidate = Path(str(commands.get('processed_dir'))).expanduser()
                if candidate.exists():
                    processed_dir = candidate

            selected_domain_id = str(commands.get('domain_id') or domain_id.value or 'science').strip() or 'science'
            selected_out_dir = Path(str(commands.get('out_dir') or out_dir.value or 'runs/task3_dual_local_blind_ab')).expanduser()
            selected_top_pairs = int(commands.get('top_pairs') or top_pairs.value)
            include_mm_value = include_multimodal.value if commands.get('include_multimodal') is None else bool(commands['include_multimodal'])
            run_vlm_value = run_vlm.value if commands.get('run_vlm') is None else bool(commands['run_vlm'])
            hf_offline_value = hf_offline.value if commands.get('hf_offline') is None else bool(commands['hf_offline'])

            if hf_offline_value:
                os.environ['HF_HUB_OFFLINE'] = '1'
                os.environ['TRANSFORMERS_OFFLINE'] = '1'
                os.environ.setdefault('HF_DATASETS_OFFLINE', '1')
            else:
                os.environ.pop('HF_HUB_OFFLINE', None)
                os.environ.pop('TRANSFORMERS_OFFLINE', None)

            model_a_cfg = {
                'owner_label': str(commands.get('model_a_owner_label') or model_a_owner_label.value or 'base_local_model').strip() or 'base_local_model',
                'vlm_backend': str(commands.get('model_a_vlm_backend') or model_a_vlm_backend.value or 'qwen2_vl').strip() or 'qwen2_vl',
                'vlm_model_id': str(commands.get('model_a_vlm_model_id') or model_a_vlm_model_id.value or '').strip(),
                'local_text_model': str(commands.get('model_a_local_text_model') or model_a_local_text_model.value or '').strip() or None,
            }
            model_b_cfg = {
                'owner_label': str(commands.get('model_b_owner_label') or model_b_owner_label.value or 'finetuned_local_model').strip() or 'finetuned_local_model',
                'vlm_backend': str(commands.get('model_b_vlm_backend') or model_b_vlm_backend.value or 'qwen2_vl').strip() or 'qwen2_vl',
                'vlm_model_id': str(commands.get('model_b_vlm_model_id') or model_b_vlm_model_id.value or '').strip(),
                'local_text_model': str(commands.get('model_b_local_text_model') or model_b_local_text_model.value or '').strip() or None,
            }

            if not model_a_cfg['vlm_model_id']:
                raise ValueError('Не указан model/path для Model α (model_a_vlm_model_id).')
            if not model_b_cfg['vlm_model_id']:
                raise ValueError('Не указан model/path для Model β (model_b_vlm_model_id).')
            if model_a_cfg['vlm_backend'] == 'none':
                run_vlm_value = False
            if model_b_cfg['vlm_backend'] == 'none':
                run_vlm_value = False

            display(Markdown(f"""
# Запуск dual-local blind A/B
- **input_mode**: `{input_mode.value}`
- **effective_query**: `{effective_query or '—'}`
- **trajectory**: `{str(trajectory_for_run) if trajectory_for_run else '—'}`
- **identifiers**: `{len(identifiers)}`
- **processed_dir (initial)**: `{str(processed_dir) if processed_dir else '—'}`
- **domain_id**: `{selected_domain_id}`
- **HF offline**: `{hf_offline_value}`
- **Model α backend**: `{model_a_cfg['vlm_backend']}`
- **Model α path/id**: `{model_a_cfg['vlm_model_id']}`
- **Model α local text**: `{model_a_cfg['local_text_model'] or '—'}`
- **Model β backend**: `{model_b_cfg['vlm_backend']}`
- **Model β path/id**: `{model_b_cfg['vlm_model_id']}`
- **Model β local text**: `{model_b_cfg['local_text_model'] or '—'}`
"""))

            common_kwargs = dict(
                trajectory=trajectory_for_run,
                query=effective_query,
                identifiers=identifiers,
                domain_id=selected_domain_id,
                search_limit=int(commands.get('search_limit') or search_limit.value),
                top_papers=int(commands.get('top_papers') or top_papers.value),
                top_hypotheses=int(commands.get('top_hypotheses') or top_hypotheses.value),
                candidate_top_k=int(commands.get('candidate_top_k') or candidate_top_k.value),
                include_multimodal=bool(include_mm_value),
                run_vlm=bool(run_vlm_value),
                edge_mode=str(commands.get('edge_mode') or edge_mode.value),
                link_prediction_backend=str(commands.get('link_prediction_backend') or link_prediction_backend.value),
                annoy_n_trees=int(annoy_n_trees.value),
                annoy_top_k=int(annoy_top_k.value),
                llm_provider=None,
                llm_model=None,
                g4f_model=None,
            )

            status.value = '<div><b>Шаг 1/3.</b> Запуск Model α ...</div>'
            bundle_alpha = prepare_task3_hypothesis_bundle(
                **common_kwargs,
                processed_dir=processed_dir,
                out_dir=selected_out_dir / 'variant_alpha',
                local_model=model_a_cfg['local_text_model'],
                vlm_backend=model_a_cfg['vlm_backend'],
                vlm_model_id=model_a_cfg['vlm_model_id'],
            )

            shared_processed_dir = processed_dir if processed_dir is not None else (Path(bundle_alpha.bundle_dir) / 'processed_papers')
            if shared_processed_dir is None or not Path(shared_processed_dir).exists():
                raise FileNotFoundError('После run α не удалось определить shared processed_papers directory.')

            status.value = '<div><b>Шаг 2/3.</b> Запуск Model β на тех же processed papers ...</div>'
            bundle_beta = prepare_task3_hypothesis_bundle(
                **common_kwargs,
                processed_dir=Path(shared_processed_dir),
                out_dir=selected_out_dir / 'variant_beta',
                local_model=model_b_cfg['local_text_model'],
                vlm_backend=model_b_cfg['vlm_backend'],
                vlm_model_id=model_b_cfg['vlm_model_id'],
            )

            manifest_alpha = json.loads(Path(bundle_alpha.manifest_path).read_text(encoding='utf-8'))
            manifest_beta = json.loads(Path(bundle_beta.manifest_path).read_text(encoding='utf-8'))

            review_assets = None
            expert_bundle_path = None
            if bool(create_offline_form.value) or bool(create_expert_bundle.value):
                status.value = '<div><b>Шаг 3/3.</b> Сборка blind offline review ...</div>'
                review_assets = build_task3_dual_model_offline_review_package(
                    manifest_alpha,
                    manifest_beta,
                    task_meta,
                    top_pairs=selected_top_pairs,
                    model_a_descriptor=model_a_cfg,
                    model_b_descriptor=model_b_cfg,
                )
            if bool(create_expert_bundle.value):
                expert_bundle_path = build_task3_dual_model_expert_bundle(
                    manifest_alpha,
                    manifest_beta,
                    task_meta,
                    top_pairs=selected_top_pairs,
                    model_a_descriptor=model_a_cfg,
                    model_b_descriptor=model_b_cfg,
                )

            paths = {
                'run_alpha_bundle': bundle_alpha.bundle_dir,
                'run_beta_bundle': bundle_beta.bundle_dir,
                'manifest_alpha': bundle_alpha.manifest_path,
                'manifest_beta': bundle_beta.manifest_path,
                'hypotheses_alpha_json': bundle_alpha.hypotheses_path,
                'hypotheses_beta_json': bundle_beta.hypotheses_path,
                'hypotheses_alpha_md': Path(bundle_alpha.bundle_dir) / 'hypotheses_ranked.md',
                'hypotheses_beta_md': Path(bundle_beta.bundle_dir) / 'hypotheses_ranked.md',
                'offline_form': review_assets.offline_html_path if review_assets else None,
                'owner_key': review_assets.owner_mapping_path if review_assets else None,
                'public_manifest': review_assets.public_manifest_path if review_assets else None,
                'expert_bundle': expert_bundle_path,
            }
            _set_task3dual_paths(paths)

            top_alpha = _top_hypotheses_summary(Path(bundle_alpha.hypotheses_path), prefix='α')
            top_beta = _top_hypotheses_summary(Path(bundle_beta.hypotheses_path), prefix='β')
            status.value = (
                '<div><b>Dual-local blind A/B завершён.</b><br>'
                f'Run α: <code>{_escape_html(str(bundle_alpha.bundle_dir))}</code><br>'
                f'Run β: <code>{_escape_html(str(bundle_beta.bundle_dir))}</code><br>'
                f'Offline HTML: <code>{_escape_html(str(review_assets.offline_html_path) if review_assets else "—")}</code><br>'
                f'Owner key: <code>{_escape_html(str(review_assets.owner_mapping_path) if review_assets else "—")}</code><br>'
                f'Expert ZIP: <code>{_escape_html(str(expert_bundle_path) if expert_bundle_path else "—")}</code>'
                '</div>'
            )
            summary_html.value = (
                '<div>'
                f'<b>Topic:</b> {_escape_html(task_meta.get("topic") or manifest_alpha.get("query") or "—")}<br>'
                f'<b>Submission:</b> <code>{_escape_html(task_meta.get("submission_id") or "—")}</code><br>'
                f'<b>Shared processed dir:</b> <code>{_escape_html(str(shared_processed_dir))}</code><br>'
                f'<b>Blind offline HTML:</b> <code>{_escape_html(str(review_assets.offline_html_path) if review_assets else "—")}</code><br>'
                f'<b>Owner key:</b> <code>{_escape_html(str(review_assets.owner_mapping_path) if review_assets else "—")}</code><br>'
                f'<b>Expert ZIP:</b> <code>{_escape_html(str(expert_bundle_path) if expert_bundle_path else "—")}</code><br>'
                '<hr><b>Top hypotheses α</b><br>' + top_alpha.replace(chr(10), '<br>') +
                '<hr><b>Top hypotheses β</b><br>' + top_beta.replace(chr(10), '<br>') +
                '</div>'
            )

            display(Markdown('## Top hypotheses — Model α'))
            display(Markdown(top_alpha))
            display(Markdown('## Top hypotheses — Model β'))
            display(Markdown(top_beta))

            if auto_download_offline.value and review_assets:
                _download_file_if_possible(Path(review_assets.offline_html_path), status_widget=status)
            if auto_download_bundle.value and expert_bundle_path:
                _download_file_if_possible(Path(expert_bundle_path), status_widget=status)
            if auto_download_owner_key.value and review_assets:
                _download_file_if_possible(Path(review_assets.owner_mapping_path), status_widget=status)

            result = {
                'bundle_alpha': bundle_alpha,
                'bundle_beta': bundle_beta,
                'review_assets': review_assets,
                'expert_bundle': expert_bundle_path,
            }
            TASK3_DUAL_LAST_RESULT = result
            return result
        except Exception as e:
            status.value = f'<div><b>Ошибка запуска dual-local blind A/B.</b><br><code>{_escape_html(type(e).__name__ + ": " + str(e))}</code></div>'
            raise
        finally:
            gc.collect()


TASK3_DUAL_LAST_RESULT = run_task3_dual_local_ab_from_form()


# Что сохраняется после запуска

В типичном случае у вас появятся:

- два отдельных Task 3 run directory (`variant_alpha/...` и `variant_beta/...`),
- blind offline HTML для эксперта,
- owner-only key file,
- expert ZIP без раскрытия owner key,
- ranked hypotheses JSON/MD для обеих модельных конфигураций.

## Практический совет
Если статьи уже были заранее скачаны и распарсены, используйте `processed_papers.zip`. Тогда второй прогон будет быстрее, а сравнение — стабильнее.


# FAQ

## Что указывать в `model/path α` и `model/path β`?
Можно указывать:
- Hugging Face model id,
- локальный путь к директории модели,
- заранее скачанную/закешированную модель.

## Когда включать `HF offline / local-files mode`?
Когда модели уже лежат локально или в кеше и вы хотите запретить сетевые обращения к Hugging Face Hub.

## Почему owner key хранится отдельно?
Потому что blind review должен скрывать от эксперта, какая анонимная система соответствует baseline и какая — finetuned модели.

## Что будет, если обе модели используют одну и ту же локальную text model?
Это допустимо. Тогда основное различие будет приходиться на VLM backend/model path и его влияние на multimodal evidence extraction и последующую формулировку гипотез.
